# Spectral Noise & Systematic Error Estimation
**Purpose:** estimate the random (photon/read) noise and the continuum-normalisation
systematic error for a rectified stellar spectrum, and write out a **single FITS
file containing both the spectrum and its wavelength-dependent 1σ error bars**.

**Output (`OUTPUT_FITS_FILE`)** — a 2-HDU FITS file:
- **Primary HDU** — continuum-normalised flux, same WCS (`CRVAL1`/`CDELT1`/`CRPIX1`) and
  identifying header keywords (`OBJNAME`/`OBJECT`, `DATE-OBS`, …) as the input rectified
  spectrum, restricted to `WL_MIN`–`WL_MAX`.
- **`ERR` extension** — the combined 1σ uncertainty on the flux (photon/read noise plus,
  optionally, the continuum-placement systematic), on the same wavelength grid.
- Header keyword **`NOISELVL`** — the scalar reference-window noise level, kept for
  downstream code that still wants a single number (e.g. a prominence floor).

Run this notebook **before** `spectral_lines_finder.ipynb`. That notebook no longer
recomputes noise itself — it loads `OUTPUT_FITS_FILE` directly and reads the spectrum
and its error bars straight out of the FITS file.

---

## 0 · Imports & configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.io import fits

plt.rcParams['figure.dpi']       = 130
plt.rcParams['axes.grid']         = True
plt.rcParams['grid.alpha']        = 0.4
plt.rcParams['grid.linestyle']    = '--'
plt.rcParams['grid.linewidth']    = 0.4
plt.rcParams['font.size']         = 13
plt.rcParams['axes.titlesize']    = 13
plt.rcParams['axes.labelsize']    = 13
plt.rcParams['xtick.labelsize']   = 11
plt.rcParams['ytick.labelsize']   = 11
plt.rcParams['legend.fontsize']   = 11
plt.rcParams['figure.titlesize']  = 14

# ── Input files ─────────────────────────────────────────────────────────────
FITS_FILE     = '_adidborealis_20251209_755-rect.fit'  # ← normalised (rectified) spectrum
RAW_FITS_FILE = '_adidborealis_20251209_755.fit'        # ← pre-normalisation (for noise scaling)

# ── Output file ───────────────────────────────────────────────────────────────
# Combined spectrum + error-bar FITS file. This is what spectral_lines_finder.ipynb
# reads instead of FITS_FILE / RAW_FITS_FILE.
OUTPUT_FITS_FILE = '_adidborealis_20251209_755-spectrum-with-errors.fit'

# ── Wavelength range ────────────────────────────────────────────────────────
# Set to None to use the full range stored in the FITS file.
# Units: Ångström
WL_MIN = 3700   # e.g. 3700  |  None → use file start
WL_MAX = 6800   # e.g. 6750  |  None → use file end

# ── Noise estimation window ──────────────────────────────────────────────────
# Choose a genuinely line-free continuum window for the star's spectral class.
# The window must be free of absorption lines AND diffuse interstellar bands
# (DIBs); contamination inflates the noise estimate and suppresses detections.
#
# Recommended windows by spectral class:
#
#   O / early B  : 5750–5850 Å  (few metallic lines; avoid He I 5876)
#   late B       : 5750–5850 Å  or  6050–6150 Å
#   A normal     : 6050–6150 Å  (avoid Na D 5890/5896 and dense Fe II forest
#                                 around 5780 Å which includes a strong DIB)
#   A supergiant : 6700–6800 Å  (5750-5850 Å is contaminated by the rich
#                                 metallic spectrum; use the red wing between
#                                 Hα 6563 and the red edge)
#   F / G        : 6100–6200 Å  (redder and cleaner than the dense green-blue)
#
# Quick sanity check: after loading, plot the flux in this window and verify
# it shows only Gaussian scatter with no obvious dips or trends.
# A good window has std/mean < 0.01 and no pixel deviating more than 3σ.
NOISE_WL_MIN = 5720   # Å — start of line-free window  ← adjust per spectral class
NOISE_WL_MAX = 5820   # Å — end   of line-free window  ← adjust per spectral class

# ── Noise rescaling ──────────────────────────────────────────────────────────
# Box width used to smooth the raw spectrum into a coarse continuum shape.
# Needs to be wide enough to wash out individual lines (~20 Å FWHM at R=850)
# but not so wide that it misses the broad instrumental response curve.
# 200 Å is a safe default; reduce to 100 Å if the response varies sharply.
NOISE_SMOOTH_WIDTH_A = 200  # A

# Power-law exponent relating noise to relative sensitivity S(λ):
#   noise(λ) = NOISE_LEVEL / S(λ)**NOISE_SCALING_POWER
# 0.5 = photon (Poisson/shot) noise -- absolute noise ∝ sqrt(counts), so
#       fractional noise after normalising ∝ 1/sqrt(counts). Correct default
#       whenever counting statistics dominate (the normal case at SNR ≳ 100).
# 1.0 = old linear behaviour -- only correct if a wavelength-independent
#       additive noise floor (read noise, dark current) dominates over shot
#       noise, which is unusual at these signal levels.
NOISE_SCALING_POWER = 0.5   # 0.5 (Poisson, recommended) | 1.0 (old linear)

# ── Continuum-normalisation systematic error ──────────────────────────────────
# The rectified spectrum (FITS_FILE) was produced by dividing the raw
# spectrum by a continuum traced *manually* (an arbitrary interpolation
# through hand-picked anchor points). That continuum estimate carries its
# own placement error, on top of the random photon/read noise already
# estimated above -- and this systematic term is often what actually shows
# up as spurious "features" in the wings of broad lines (e.g. Balmer),
# where a slightly mismatched continuum leaves a broad, smooth residual
# that a Voigt fit cannot absorb. Section 4 reconstructs that continuum
# (raw / normalised) and measures how far it wanders from a local straight
# line using a *moving* window -- CONT_ERR_WINDOW_A wide, re-centred every
# CONT_ERR_STEP_A -- rather than a fixed, non-overlapping bin grid, so the
# resulting fractional error varies smoothly with wavelength instead of
# jumping at hard bin edges. The result is folded into noise_spectrum in
# quadrature.
CONT_ERR_WINDOW_A   = 50      # Angstrom -- width of the moving local linear-fit window
CONT_ERR_STEP_A     = 10      # Angstrom -- distance the window is re-centred by at each step (< window → overlapping)
CONT_ERR_SIGMA_CLIP = 2.5     # sigma -- clip points this far from the local line (rejects real absorption/emission lines, not continuum error)
CONT_ERR_NITER      = 3       # iterative sigma-clipping passes per window
CONT_ERR_MIN_PTS    = 8       # minimum surviving (line-free) points required to trust a window's fit
ADD_CONTINUUM_ERROR = True    # master switch -- False reproduces the old photon-noise-only behaviour

## 1 · Load the FITS file
Uses the standard FITS WCS keywords `CRVAL1`, `CDELT1`, `CRPIX1` to reconstruct the wavelength array.

In [ ]:
def load(file_name):
    """Load a 1-D spectrum from a Visual Spec FITS file.

    Returns
    -------
    wavelength : ndarray  [Å]
    flux       : ndarray  [normalised]
    header     : fits.Header
    """
    hdul   = fits.open(file_name)
    header = hdul[0].header
    flux   = hdul[0].data.astype(float)   # convert big-endian float32 → float64

    crval = header.get('CRVAL1')           # reference wavelength [Å]
    cdelt = header.get('CDELT1')           # Å per pixel
    crpix = header.get('CRPIX1', 1)        # reference pixel (1-based)
    npix  = flux.size
    pixel = np.arange(npix)               # 0-based pixel index
    wavelength = crval + (pixel + 1 - crpix) * cdelt

    return wavelength, flux, header


def get_object_name(header, fallback_file=None, default='Unknown target'):
    """Extract the star's name from a FITS header.

    Tries the OBJNAME keyword first (used by ISIS/BSS-format raw spectra),
    then OBJECT (a common alternative), then — if the rectified file's header
    doesn't carry either (this is normal: normalisation often strips them) —
    peeks at fallback_file's header instead.  Falls back to a generic label
    if no identifying keyword is found anywhere, so the notebook still runs
    on files with minimal headers.
    """
    for key in ('OBJNAME', 'OBJECT'):
        name = header.get(key)
        if name:
            return str(name).strip()
    if fallback_file is not None:
        try:
            fb_header = fits.open(fallback_file)[0].header
            for key in ('OBJNAME', 'OBJECT'):
                name = fb_header.get(key)
                if name:
                    return str(name).strip()
        except Exception:
            pass
    return default


wavelength, flux, header = load(FITS_FILE)

def get_obs_date(header, fallback_file=None, default=None):
    """Extract the observation date (DATE-OBS) the same way as get_object_name."""
    date = header.get('DATE-OBS')
    if date:
        return str(date).split('T')[0]   # keep date only, drop time-of-day
    if fallback_file is not None:
        try:
            fb_header = fits.open(fallback_file)[0].header
            date = fb_header.get('DATE-OBS')
            if date:
                return str(date).split('T')[0]
        except Exception:
            pass
    return default


# Star name and observation date: check this file's header first, then fall\
# back to RAW_FITS_FILE's header (normalised/rectified spectra often have\
# these keywords stripped during processing, while the original raw\
# acquisition file usually keeps them).
OBJECT_NAME = get_object_name(header, fallback_file=RAW_FITS_FILE)
OBS_DATE    = get_obs_date(header, fallback_file=RAW_FITS_FILE)

print(f"Object    : {OBJECT_NAME}")
print(f"Obs. date : {OBS_DATE if OBS_DATE else 'n/a'}")
print(f"Generator : {header.get('COMMENT', 'n/a')}")
print(f"Pixels    : {flux.size}")
print(f"Dispersion: {header['CDELT1']:.4f} Å / px")
print(f"Full range: {wavelength[0]:.2f} – {wavelength[-1]:.2f} Å")

# ── Apply wavelength limits ──────────────────────────────────────────────────
lo = WL_MIN if WL_MIN is not None else wavelength[0]
hi = WL_MAX if WL_MAX is not None else wavelength[-1]
mask = (wavelength >= lo) & (wavelength <= hi)
wavelength, flux = wavelength[mask], flux[mask]

print(f"Working range: {wavelength[0]:.2f} – {wavelength[-1]:.2f} Å  ({wavelength.size} px)")
print(f"Flux      : min={flux.min():.4f}  max={flux.max():.4f}  mean={flux.mean():.4f}")


## 2 · Noise estimation
We estimate the noise from a **line-free continuum window** (`NOISE_WL_MIN` / `NOISE_WL_MAX`).  
Three complementary metrics are computed:

| Metric | Formula | Meaning |
|--------|---------|----------|
| **σ (std dev)** | `std(flux_window)` | RMS scatter around the local mean — best general noise estimator |
| **NMAD** | `1.4826 × median(|f − median(f)|)` | Robust to residual cosmic rays or weak lines |
| **SNR** | `mean(flux_window) / σ` | Signal-to-noise ratio in the window |

The result is used later as the detection threshold for absorption lines.

In [ ]:
# ── Noise estimation ─────────────────────────────────────────────────────────
noise_mask   = (wavelength >= NOISE_WL_MIN) & (wavelength <= NOISE_WL_MAX)
noise_pixels = noise_mask.sum()

if noise_pixels < 5:
    raise ValueError(
        f'Noise window {NOISE_WL_MIN}–{NOISE_WL_MAX} Å contains only '
        f'{noise_pixels} pixel(s). Widen the window or move it inside '
        f'the working range {WL_MIN}–{WL_MAX} Å.'
    )

flux_win  = flux[noise_mask]
wl_win    = wavelength[noise_mask]

noise_std  = flux_win.std(ddof=1)                          # sample std dev
noise_nmad = 1.4826 * np.median(np.abs(flux_win - np.median(flux_win)))  # robust
noise_mean = flux_win.mean()
snr_window = noise_mean / noise_std

# Store the primary noise estimate used downstream (std dev)
NOISE_LEVEL = noise_std

print('Noise estimation window')
print(f'  Range   : {wl_win[0]:.2f} – {wl_win[-1]:.2f} Å  ({noise_pixels} px)')
print(f'  Mean    : {noise_mean:.5f}')
print(f'  Std dev : {noise_std:.5f}   ← NOISE_LEVEL used downstream')
print(f'  NMAD    : {noise_nmad:.5f}   (robust estimate)')
print(f'  SNR     : {snr_window:.1f}')

# ── Inset plot of the noise window ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5),
                         gridspec_kw={'width_ratios': [3, 1]})

# Left panel: zoom on the window in context
ctx = 200   # Å of context on each side
ctx_lo = max(wavelength[0], NOISE_WL_MIN - ctx)
ctx_hi = min(wavelength[-1], NOISE_WL_MAX + ctx)
ctx_mask = (wavelength >= ctx_lo) & (wavelength <= ctx_hi)

ax0 = axes[0]
ax0.plot(wavelength[ctx_mask], flux[ctx_mask],
         color='steelblue', linewidth=0.9, label='Spectrum')
ax0.axvspan(NOISE_WL_MIN, NOISE_WL_MAX, color='gold', alpha=0.35, label='Noise window')
ax0.axhline(noise_mean,             color='black',  linewidth=1.2, linestyle='-',  label='Mean')
ax0.axhline(noise_mean + noise_std, color='tomato', linewidth=1.1, linestyle='--', label=r'Mean ± 1$\sigma$')
ax0.axhline(noise_mean - noise_std, color='tomato', linewidth=1.1, linestyle='--')
ax0.set_xlabel('Wavelength (Å)', fontsize=13)
ax0.set_ylabel('Normalised Flux', fontsize=13)
ax0.set_title(f'Noise window in context  [{NOISE_WL_MIN}–{NOISE_WL_MAX} Å]', fontsize=13)
ax0.legend(fontsize=11)

# Right panel: flux histogram of the noise window + Gaussian overlay
ax1 = axes[1]
counts, bin_edges, _ = ax1.hist(flux_win, bins=20, color='gold',
                                edgecolor='goldenrod', alpha=0.8,
                                orientation='vertical', density=True)
xs = np.linspace(flux_win.min(), flux_win.max(), 300)
gauss = (1 / (noise_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((xs - noise_mean) / noise_std) ** 2)
ax1.plot(xs, gauss, color='tomato', linewidth=1.8, label='Gaussian fit')
ax1.set_xlabel('Flux value', fontsize=13)
ax1.set_ylabel('Density', fontsize=13)
ax1.set_title('Pixel distribution', fontsize=13)
ax1.legend(fontsize=11)
ax1.text(0.05, 0.92, f'$\\sigma$ = {noise_std:.5f}\nSNR = {snr_window:.0f}',
         transform=ax1.transAxes, fontsize=11, va='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

## 3 · Wavelength-dependent noise rescaling
The scalar `NOISE_LEVEL` from Section 2 is valid only at the reference window (~5750 Å).  
Elsewhere the raw counts are very different — the instrumental response and the stellar SED together produce a large flux variation across the working range — so the noise in the normalised spectrum also varies.

**Strategy (coarse-grained, no line-fitting needed):**
1. Load the pre-normalisation raw spectrum.
2. Smooth it with a wide box filter (`NOISE_SMOOTH_WIDTH_A`) to obtain a coarse continuum — this washes out all individual lines while preserving the large-scale sensitivity shape.
3. Normalise that continuum to 1.0 at the noise reference window → **relative sensitivity curve** `S(λ)`.
4. Scale the noise by a power law in `S(λ)`:
   $$\text{noise}(\lambda) = \frac{\text{NOISE\_LEVEL}}{S(\lambda)^{\,p}}, \qquad p = \text{NOISE\_SCALING\_POWER}$$
   **`p = 0.5` (default) — photon (shot) noise.** Counting statistics are Poisson:
   the absolute noise in the raw counts scales as $\sqrt{N}$, so after dividing
   by a continuum $C(\lambda)\!\approx\!N(\lambda)$ the *fractional* noise in the
   normalised spectrum scales as $\sigma_{\text{norm}} \propto \sqrt{N}/N = 1/\sqrt{N}$
   — an inverse **square-root** law in the count level, i.e. in $S(\lambda)$.
   `p = 1.0` (the old behaviour, `noise ∝ 1/S(λ)`) would only be correct if a
   wavelength-independent additive noise floor (read noise, dark current)
   dominated over shot noise — unusual once the reference-window SNR is in the
   hundreds, as it is here.

The result `noise_spectrum` replaces `NOISE_LEVEL` everywhere downstream.

In [ ]:
from scipy.ndimage import uniform_filter1d

# ── Load the raw (pre-normalisation) spectrum ─────────────────────────────
wl_raw, fl_raw, _ = load(RAW_FITS_FILE)
m_raw = (wl_raw >= WL_MIN) & (wl_raw <= WL_MAX)
wl_raw, fl_raw = wl_raw[m_raw], fl_raw[m_raw]
print(f'Raw spectrum loaded: {wl_raw[0]:.1f} – {wl_raw[-1]:.1f} Å  ({wl_raw.size} px)')
print(f'Raw flux range: {fl_raw.min():.2f} – {fl_raw.max():.2f}')

# ── Coarse continuum by box smoothing ────────────────────────────────────
cdelt_raw  = np.median(np.diff(wl_raw))
width_px   = max(3, int(NOISE_SMOOTH_WIDTH_A / cdelt_raw))
continuum  = uniform_filter1d(fl_raw, size=width_px)
print(f'\nSmoothing width: {NOISE_SMOOTH_WIDTH_A} Å = {width_px} px')

# ── Reference level at the noise window ──────────────────────────────────
nm_raw  = (wl_raw >= NOISE_WL_MIN) & (wl_raw <= NOISE_WL_MAX)
ref_val = np.median(continuum[nm_raw])
print(f'Reference continuum level at noise window: {ref_val:.3f}')

# ── Relative sensitivity S(λ) ────────────────────────────────────────────
rel_sensitivity = continuum / ref_val   # = 1.0 at the noise window
print(f'Sensitivity range: {rel_sensitivity.min():.3f} – {rel_sensitivity.max():.3f}')

# ── Interpolate S(λ) onto the normalised spectrum grid ───────────────────
rel_sens_nrm = np.interp(wavelength, wl_raw, rel_sensitivity)

# ── Wavelength-dependent noise array ────────────────────────────────────
# More raw counts → lower fractional noise in the normalised spectrum, via a
# power law in the relative sensitivity: noise(λ) = NOISE_LEVEL / S(λ)**p.
# p = NOISE_SCALING_POWER (0.5 = photon/Poisson noise, the physically-motivated
# default; 1.0 reproduces the old linear behaviour -- see Section 3 markdown).
noise_spectrum = NOISE_LEVEL / rel_sens_nrm**NOISE_SCALING_POWER

# Sanity check: value at the reference window should equal NOISE_LEVEL
check_wl   = (NOISE_WL_MIN + NOISE_WL_MAX) / 2
check_val  = float(np.interp(check_wl, wavelength, noise_spectrum))
print(f'\nSanity check at {check_wl:.0f} Å: noise_spectrum = {check_val:.5f}'
      f'  (NOISE_LEVEL = {NOISE_LEVEL:.5f})')

print('\nnoise_spectrum at key wavelengths:')
for wl_check in [3800, 4340, 4861, 5750, 6563]:
    if wavelength[0] <= wl_check <= wavelength[-1]:
        val = float(np.interp(wl_check, wavelength, noise_spectrum))
        ratio = val / NOISE_LEVEL
        print(f'  {wl_check} Å → {val:.5f}  ({ratio:.2f}× reference)')

# ── Three-panel diagnostic figure ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(20, 14), sharex=False)

# Panel 1: raw spectrum + smoothed continuum
ax = axes[0]
ax.plot(wl_raw, fl_raw, color='steelblue', linewidth=0.6, alpha=0.6, label='Raw flux')
ax.plot(wl_raw, continuum, color='tomato', linewidth=2.2,
        label=f'Smoothed continuum ({NOISE_SMOOTH_WIDTH_A} Å box)')
ax.axvspan(NOISE_WL_MIN, NOISE_WL_MAX, color='gold', alpha=0.4, label='Noise window')
ax.axhline(ref_val, color='goldenrod', linewidth=1.2, linestyle='--',
           label=f'Reference level ({ref_val:.2f})')
ax.set_ylabel('Raw flux (ADU)', fontsize=13)
ax.set_title('Panel 1 — Raw spectrum and coarse continuum envelope', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(WL_MIN, WL_MAX)
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(50))

# Panel 2: relative sensitivity S(λ)
ax = axes[1]
ax.plot(wl_raw, rel_sensitivity, color='darkorange', linewidth=1.8)
ax.axhline(1.0, color='black', linewidth=1.0, linestyle='--', label='Reference = 1')
ax.axvspan(NOISE_WL_MIN, NOISE_WL_MAX, color='gold', alpha=0.4, label='Noise window')
ax.set_ylabel('Relative sensitivity  S(λ)', fontsize=13)
ax.set_title('Panel 2 — Relative sensitivity (continuum / reference level)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(WL_MIN, WL_MAX)
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(50))

# Panel 3: noise_spectrum vs flat NOISE_LEVEL
ax = axes[2]
ax.plot(wavelength, noise_spectrum, color='seagreen', linewidth=1.8,
        label='noise_spectrum (wavelength-dependent)')
ax.axhline(NOISE_LEVEL, color='black', linewidth=1.1, linestyle='--',
           label=f'Scalar NOISE_LEVEL = {NOISE_LEVEL:.4f}')
ax.axvspan(NOISE_WL_MIN, NOISE_WL_MAX, color='gold', alpha=0.4, label='Reference window')
ax.set_xlabel('Wavelength (Å)', fontsize=13)
ax.set_ylabel('Noise (norm. flux units)', fontsize=13)
ax.set_title('Panel 3 — Wavelength-dependent noise  [noise_spectrum used downstream]', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(WL_MIN, WL_MAX)
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(50))

plt.tight_layout()
plt.show()

## 4 · Continuum-normalisation systematic error

Section 3 rescaled the **random** (photon/read) noise across wavelength. But the
rectified spectrum was built by dividing the raw spectrum by a continuum that was
traced **manually** — an arbitrary interpolation through hand-picked anchor points.
That continuum is not perfectly known, and its placement error is a **systematic**
uncertainty, independent of the random noise above. This is very likely the source
of the spurious features seen in the wings of broad lines (Balmer, in particular):
a slightly mismatched continuum leaves a broad, smooth mismatch in the wings that
no Voigt profile can reproduce, and the leftover residual looks exactly like a
faint extra absorption feature to a peak finder that only knows about photon noise.

**Procedure:**
1. **Reconstruct the continuum actually used** for the normalisation:
   $$C(\lambda) = \frac{\text{raw}(\lambda)}{\text{normalised}(\lambda)}$$
   (raw spectrum interpolated onto the rectified spectrum's wavelength grid).
2. **Local linear fit in a moving window**, `CONT_ERR_WINDOW_A` Å wide (50 Å by
   default), re-centred every `CONT_ERR_STEP_A` Å (10 Å by default) so
   consecutive windows overlap. A real continuum varies smoothly on these
   scales, so a straight line is a fair local model of "what the continuum
   should look like" inside each window; deep absorption lines are excluded
   from each window's fit via iterative `CONT_ERR_SIGMA_CLIP`-σ clipping so
   they don't bias the linear trend. Sliding the window rather than stepping
   through fixed, non-overlapping bins avoids the hard jumps in $\sigma_C/C$
   that a bin grid produces right at each bin edge — the estimate is
   continuous and re-evaluated at every step instead of shared piecewise
   across a whole 50 Å block.
3. **Residuals** of $C(\lambda)$ from its local linear fit, evaluated at the
   window's *centre* and divided by the local continuum level there, give the
   *fractional* continuum-placement uncertainty $\sigma_C/C$ at that centre.
   The per-centre values are then linearly interpolated onto the full
   per-pixel wavelength grid.
4. **Propagate to the normalised flux.** Since $\text{normalised} = \text{raw}/C$,
   standard error propagation gives
   $$\sigma_{\text{continuum}}(\lambda) \approx \text{flux}(\lambda)\times\frac{\sigma_C(\lambda)}{C(\lambda)}$$
   which is added **in quadrature** to the photon-noise `noise_spectrum(λ)` from
   Section 3:
   $$\sigma_{\text{total}}(\lambda) = \sqrt{\sigma_{\text{photon}}^2(\lambda) + \sigma_{\text{continuum}}^2(\lambda)}$$

`noise_spectrum` and the scalar `NOISE_LEVEL` are both updated in place, so every
downstream step (detection height/prominence, deblending thresholds, SSR_norm)
automatically uses the combined uncertainty. Set `ADD_CONTINUUM_ERROR = False` in
the config cell to fall back to the old photon-noise-only behaviour.


In [ ]:
# ── 1. Reconstruct the continuum that was actually used ─────────────────────
# raw / normalised, on the rectified spectrum's wavelength grid (raw is
# interpolated since the two files don't share the exact same pixel phase).
raw_on_nrm_grid = np.interp(wavelength, wl_raw, fl_raw)
with np.errstate(divide='ignore', invalid='ignore'):
    continuum_reconstructed = np.where(flux > 0, raw_on_nrm_grid / flux, np.nan)

# ── 2 & 3. Moving-window robust linear fit + fractional residual RMS ────────
# A fixed, non-overlapping bin grid produces a hard jump in sigma_C/C right at
# every bin edge, purely from where the grid happened to be laid down -- not
# from anything physical in the continuum. Instead, slide a CONT_ERR_WINDOW_A
# wide window across the spectrum, re-centring it every CONT_ERR_STEP_A (so
# neighbouring windows overlap), fit + sigma-clip within each window exactly
# as before, and evaluate the fractional residual at the window's *centre*.
# The result is a dense, smoothly-varying grid of (centre, sigma_C/C) pairs
# that is then interpolated onto the full per-pixel wavelength array.
half_win = CONT_ERR_WINDOW_A / 2.0
win_centers = np.arange(wavelength[0], wavelength[-1] + CONT_ERR_STEP_A, CONT_ERR_STEP_A)
win_centers = win_centers[(win_centers >= wavelength[0]) & (win_centers <= wavelength[-1])]

frac_centers  = np.full(win_centers.shape, np.nan)   # fractional sigma_C/C, at each window centre
level_centers = np.full(win_centers.shape, np.nan)   # local linear-fit value at the centre, for diagnostics

win_report = []
for i, wc in enumerate(win_centers):
    w0, w1 = wc - half_win, wc + half_win
    m_win = (wavelength >= w0) & (wavelength <= w1) & np.isfinite(continuum_reconstructed)
    n_win = int(m_win.sum())
    if n_win < CONT_ERR_MIN_PTS:
        win_report.append((wc, n_win, 0, np.nan))
        continue

    x_win = wavelength[m_win]
    y_win = continuum_reconstructed[m_win]

    # Pre-filter pass: reject gross outliers from dividing by near-zero flux
    # in deep line cores, using a robust median/MAD estimate (insensitive to
    # the very outliers we're trying to remove) before any least-squares fit.
    med = np.median(y_win)
    mad = np.median(np.abs(y_win - med)) * 1.4826 + 1e-12
    keep = np.abs(y_win - med) <= 8 * mad

    # Iterative sigma-clipped linear fit: real absorption/emission lines pull
    # points away from the smooth continuum trend and get clipped out, leaving
    # only the continuum's own point-to-point wobble in the residuals.
    for _ in range(CONT_ERR_NITER):
        if keep.sum() < CONT_ERR_MIN_PTS:
            break
        p = np.polyfit(x_win[keep], y_win[keep], 1)
        resid_all = y_win - np.polyval(p, x_win)
        sigma = resid_all[keep].std(ddof=1) if keep.sum() > 1 else np.nan
        if not np.isfinite(sigma) or sigma == 0:
            break
        new_keep = np.abs(resid_all) <= CONT_ERR_SIGMA_CLIP * sigma
        if new_keep.sum() == keep.sum():
            keep = new_keep
            break
        keep = new_keep

    if keep.sum() < CONT_ERR_MIN_PTS:
        win_report.append((wc, n_win, int(keep.sum()), np.nan))
        continue

    p_final     = np.polyfit(x_win[keep], y_win[keep], 1)
    resid_final = (y_win - np.polyval(p_final, x_win))[keep]
    rms         = resid_final.std(ddof=1)
    local_level = np.polyval(p_final, wc)   # local trend evaluated AT the window centre
    frac        = rms / local_level if local_level != 0 else np.nan

    frac_centers[i]  = frac
    level_centers[i] = local_level
    win_report.append((wc, n_win, int(keep.sum()), frac))

# Fill any window centres that failed (too few points, edge effects) by
# interpolating from neighbouring valid centres, then interpolate the whole
# thing onto the full per-pixel wavelength grid -- no gaps either way.
valid = np.isfinite(frac_centers)
if valid.sum() >= 2:
    frac_centers  = np.interp(win_centers, win_centers[valid], frac_centers[valid])
    level_centers = np.interp(win_centers, win_centers[valid], level_centers[valid])
elif valid.sum() == 1:
    frac_centers[:]  = frac_centers[valid][0]
    level_centers[:] = level_centers[valid][0]
else:
    frac_centers[:]  = 0.0
    level_centers[:] = np.nanmedian(continuum_reconstructed)

cont_err_frac  = np.interp(wavelength, win_centers, frac_centers)
cont_fit_curve = np.interp(wavelength, win_centers, level_centers)

# ── 4. Propagate to normalised-flux uncertainty, combine in quadrature ──────
continuum_noise_spectrum = np.abs(flux * cont_err_frac)

noise_spectrum_photon_only = noise_spectrum.copy()   # keep Section 3's estimate for comparison
NOISE_LEVEL_photon_only    = NOISE_LEVEL

if ADD_CONTINUUM_ERROR:
    noise_spectrum = np.sqrt(noise_spectrum_photon_only**2 + continuum_noise_spectrum**2)
    cont_err_at_ref = float(np.interp((NOISE_WL_MIN + NOISE_WL_MAX) / 2, wavelength, continuum_noise_spectrum))
    NOISE_LEVEL = float(np.sqrt(NOISE_LEVEL_photon_only**2 + cont_err_at_ref**2))
else:
    print('ADD_CONTINUUM_ERROR = False -> noise_spectrum left as photon-noise-only (Section 3 values).')

# ── Reporting ────────────────────────────────────────────────────────────────
n_total = len(win_report)
n_ok    = sum(1 for *_, f in win_report if np.isfinite(f))
print(f'Continuum-error windows : {CONT_ERR_WINDOW_A} A wide, {CONT_ERR_STEP_A} A step, '
      f'{n_total} centres, {n_ok} fitted successfully')
print(f'                           (>= {CONT_ERR_MIN_PTS} line-free px after {CONT_ERR_NITER}x {CONT_ERR_SIGMA_CLIP} sigma clip)')
print(f'Fractional cont. error  sigma_C/C : median={np.nanmedian(cont_err_frac):.5f}  '
      f'range={np.nanmin(cont_err_frac):.5f} - {np.nanmax(cont_err_frac):.5f}')
print(f'NOISE_LEVEL (reference window): {NOISE_LEVEL_photon_only:.5f} (photon only) -> {NOISE_LEVEL:.5f} (combined)')

print(f'\nnoise_spectrum at key wavelengths  [photon-only  ->  continuum-only  ->  combined]:')
for wl_check in [3800, 4340, 4861, 5750, 6563]:
    if wavelength[0] <= wl_check <= wavelength[-1]:
        i = int(np.argmin(np.abs(wavelength - wl_check)))
        print(f'  {wl_check:5d} A :  {noise_spectrum_photon_only[i]:.5f}  ->  '
              f'{continuum_noise_spectrum[i]:.5f}  ->  {noise_spectrum[i]:.5f}')

print('\nWorst (largest fractional continuum error) window centres:')
worst = sorted([w for w in win_report if np.isfinite(w[3])], key=lambda w: -w[3])[:5]
for wc, n_win, n_keep, frac in worst:
    print(f'  centre {wc:7.1f} A   n={n_win:4d}  kept={n_keep:4d}   sigma_C/C = {frac:.5f}')


In [ ]:
# Reference wavelengths used only for annotating the diagnostic figure below
# (kept local to this notebook -- the finder notebook has its own BALMER_LINES
# used for classification, defined in its config cell).
LANDMARK_LINES = {
    r'H$\epsilon$ 3970': 3970.07,
    r'H$\delta$ 4102':   4101.74,
    r'H$\gamma$ 4340':   4340.47,
    r'H$\beta$ 4861':    4861.33,
    r'Na D 5893':        5892.94,
    r'H$\alpha$ 6563':   6562.80,
}
BALMER_LINES = {
    'Hepsilon': 3970.07,
    'Hdelta':   4101.74,
    'Hgamma':   4340.47,
    'Hbeta':    4861.33,
    'Halpha':   6562.80,
}
DETECTION_NSIGMA = 3  # matches the default in spectral_lines_finder.ipynb; only used
                       # to size the shaded detection band in the zoom plot below
C_SPEC = '#2c3e50'    # dark grey -- spectrum trace (matches finder notebook's colour)

# ── Diagnostic figure: reconstructed continuum + noise budget ───────────────
fig, axes = plt.subplots(3, 1, figsize=(20, 14), sharex=False)

# Panel 1: reconstructed continuum with the moving-window local linear fits
ax = axes[0]
ax.plot(wavelength, continuum_reconstructed, color='steelblue', linewidth=0.7,
        alpha=0.75, label='Reconstructed continuum  C(λ) = raw / normalised')
ax.plot(wavelength, cont_fit_curve, color='tomato', linewidth=1.6,
        label=f'Moving-window local fits ({CONT_ERR_WINDOW_A} Å wide, {CONT_ERR_STEP_A} Å step, {CONT_ERR_SIGMA_CLIP}σ clipped)')
ax.set_ylabel('Raw counts (ADU)', fontsize=13)
ax.set_title('Panel 1 — Reconstructed continuum vs. its moving-window linear-fit trend', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(WL_MIN, WL_MAX)
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(50))

# Panel 2: fractional continuum-placement error σ_C/C
ax = axes[1]
ax.plot(wavelength, cont_err_frac, color='darkorange', linewidth=1.8)
for wl_l, label in LANDMARK_LINES.items():
    if wavelength[0] <= label <= wavelength[-1]:
        ax.axvline(label, color='tomato', linewidth=0.9, linestyle='--', alpha=0.5)
ax.set_ylabel(r'Fractional error  $\sigma_C/C$', fontsize=13)
ax.set_title('Panel 2 — Continuum-placement fractional uncertainty (dashed lines = Balmer/Na D)', fontsize=13)
ax.set_xlim(WL_MIN, WL_MAX)
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(50))

# Panel 3: noise budget — photon-only vs continuum-only vs combined
ax = axes[2]
ax.plot(wavelength, noise_spectrum_photon_only, color='seagreen', linewidth=1.4,
        alpha=0.8, label='Photon noise only (Section 3)')
ax.plot(wavelength, continuum_noise_spectrum, color='darkorange', linewidth=1.4,
        alpha=0.8, label='Continuum-placement error (Section 4)')
ax.plot(wavelength, noise_spectrum, color='black', linewidth=1.9,
        label='Combined (quadrature sum) — used downstream')
ax.set_xlabel('Wavelength (Å)', fontsize=13)
ax.set_ylabel('Noise (norm. flux units)', fontsize=13)
ax.set_title('Panel 3 — Noise budget: photon vs. continuum-placement vs. combined', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(WL_MIN, WL_MAX)
ax.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(50))

plt.tight_layout()
plt.show()

# ── Zoom on a Balmer wing to show the effect directly ────────────────────────
zoom_line = min(BALMER_LINES.items(), key=lambda kv: 0 if wavelength[0] <= kv[1] <= wavelength[-1] else np.inf)
wl_zoom = zoom_line[1]
if wavelength[0] <= wl_zoom <= wavelength[-1]:
    half_win = 60  # Å
    m_zoom = (wavelength >= wl_zoom - half_win) & (wavelength <= wl_zoom + half_win)

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(wavelength[m_zoom], flux[m_zoom], color=C_SPEC, linewidth=1.1, label='Spectrum')
    ax.fill_between(wavelength[m_zoom],
                     1.0 - DETECTION_NSIGMA * noise_spectrum_photon_only[m_zoom],
                     1.0 + DETECTION_NSIGMA * noise_spectrum_photon_only[m_zoom],
                     color='seagreen', alpha=0.25, label=f'±{DETECTION_NSIGMA}σ — photon noise only (old)')
    ax.fill_between(wavelength[m_zoom],
                     1.0 - DETECTION_NSIGMA * noise_spectrum[m_zoom],
                     1.0 + DETECTION_NSIGMA * noise_spectrum[m_zoom],
                     color='black', alpha=0.12, label=f'±{DETECTION_NSIGMA}σ — combined (new)')
    ax.set_xlabel('Wavelength (Å)', fontsize=13)
    ax.set_ylabel('Normalised flux', fontsize=13)
    ax.set_title(f'{zoom_line[0]} {wl_zoom:.1f} Å wing — detection band before/after continuum-error term', fontsize=13)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()


## 5 · Save spectrum + error bars to a combined FITS file
Writes `OUTPUT_FITS_FILE`, a 2-HDU FITS file:

- **Primary HDU** — `flux` (continuum-normalised spectrum), with a WCS rebuilt so that
  pixel 1 corresponds to `wavelength[0]` (the array has already been trimmed to
  `WL_MIN`–`WL_MAX`), plus the identifying keywords (`OBJNAME`/`OBJECT`, `DATE-OBS`)
  and a `NOISELVL` keyword carrying the scalar reference-window noise level.
- **`ERR` extension** — `noise_spectrum`, the wavelength-dependent 1σ uncertainty
  (photon noise, optionally combined in quadrature with the continuum-placement
  systematic), on the exact same wavelength grid as the primary HDU.

`spectral_lines_finder.ipynb` reads both arrays straight out of this file.

In [ ]:
# ── Build the primary HDU: flux + WCS + identifying keywords ────────────────
primary_header = header.copy()

# Rebuild the WCS so pixel 1 == wavelength[0] (the arrays were trimmed to
# WL_MIN-WL_MAX above; CRVAL1/CRPIX1 from the original file no longer apply).
primary_header['CRVAL1'] = float(wavelength[0])
primary_header['CRPIX1'] = 1
primary_header['CDELT1'] = float(np.median(np.diff(wavelength)))

# Make sure the object name / obs date survive even if the rectified file's
# own header was missing them (get_object_name/get_obs_date already fell
# back to RAW_FITS_FILE for this above).
primary_header['OBJNAME']  = OBJECT_NAME
if OBS_DATE:
    primary_header['DATE-OBS'] = OBS_DATE

# Carry the noise-estimation bookkeeping along for downstream reference /
# reproducibility, even though the array itself lives in the ERR extension.
primary_header['NOISELVL'] = (float(NOISE_LEVEL), '1-sigma noise at the reference window')
primary_header['NSEWLMIN'] = (float(NOISE_WL_MIN), 'Noise reference window lower bound [A]')
primary_header['NSEWLMAX'] = (float(NOISE_WL_MAX), 'Noise reference window upper bound [A]')
primary_header['CONTERR']  = (bool(ADD_CONTINUUM_ERROR), 'Continuum-placement systematic folded into ERR?')
primary_header['HISTORY']  = 'Spectrum + noise_spectrum written by spectral_noise.ipynb'

primary_hdu = fits.PrimaryHDU(data=flux.astype(np.float32), header=primary_header)

# ── Build the ERR extension: same WCS, no need for the extra bookkeeping ────
err_header = fits.Header()
err_header['CRVAL1'] = primary_header['CRVAL1']
err_header['CRPIX1'] = primary_header['CRPIX1']
err_header['CDELT1'] = primary_header['CDELT1']
err_header['BUNIT']  = 'Normalised flux (1-sigma)'
err_hdu = fits.ImageHDU(data=noise_spectrum.astype(np.float32), header=err_header, name='ERR')

hdul_out = fits.HDUList([primary_hdu, err_hdu])
hdul_out.writeto(OUTPUT_FITS_FILE, overwrite=True)

print(f'Wrote {OUTPUT_FITS_FILE}')
print(f'  Primary HDU (flux)           : {flux.size} px,  {wavelength[0]:.2f}-{wavelength[-1]:.2f} A')
print(f'  ERR extension (noise_spectrum): {noise_spectrum.size} px')
print(f'  NOISELVL (reference-window sigma): {NOISE_LEVEL:.5f}')
print(f'  Continuum-placement systematic folded in: {ADD_CONTINUUM_ERROR}')
